# Keşifsel veri analizi (B1.4)

Girdi: `greenchoice build --source interim --eprel data/raw/v4/eprel_manual.csv` ile üretilen
`data/processed/greenchoice.db` (80 ürün, kural tabanlı aspect çıkarımı). Grafikler
`docs/figures/` altına kaydedilir. Defter yalnızca sayı ve oran yazdırır; yorum metni çıktıya girmez.

In [1]:
import sqlite3
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FIG = ROOT / "docs" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
con = sqlite3.connect(ROOT / "data" / "processed" / "greenchoice.db")
products = pd.read_sql("select * from products", con)
reviews = pd.read_sql("select review_id, product_id, rating from reviews", con)
aspects = pd.read_sql("select review_id, aspect, polarity from review_aspects", con)
CAT = {"refrigerator": "Buzdolabı", "washing_machine": "Çamaşır makinesi"}
products["kategori"] = products["category"].map(CAT)
reviews = reviews.merge(products[["product_id", "kategori", "source"]], on="product_id")
print(len(products), "ürün |", len(reviews), "yorum |", len(aspects), "aspect eşleşmesi")

80 ürün | 3383 yorum | 787 aspect eşleşmesi


## Eksik veri

In [2]:
cols = ["price_tl", "energy_class", "energy_kwh_per_year", "energy_kwh_per_100_cycles",
        "water_l_per_cycle", "capacity", "noise_db", "warranty_years", "rating_avg", "rating_count"]
missing = products.groupby("kategori")[cols].apply(lambda g: g.isna().mean().mul(100).round(0)).T
missing.columns.name = None
print("Eksik oranı (%)")
print(missing.to_string())
print()
print("EPREL eşleşmesi:", int(products["eprel_matched"].sum()), "/", len(products))

Eksik oranı (%)
                           Buzdolabı  Çamaşır makinesi
price_tl                         0.0               0.0
energy_class                     2.0               2.0
energy_kwh_per_year             12.0             100.0
energy_kwh_per_100_cycles      100.0              10.0
water_l_per_cycle              100.0              10.0
capacity                        50.0              10.0
noise_db                        50.0              30.0
warranty_years                 100.0             100.0
rating_avg                       5.0               0.0
rating_count                     5.0               0.0

EPREL eşleşmesi: 48 / 80


## Fiyat

In [3]:
print(products.groupby("kategori")["price_tl"].describe().round(0).to_string())
fig, ax = plt.subplots(figsize=(6, 3.2))
data = [products.loc[products["kategori"] == k, "price_tl"] / 1000 for k in CAT.values()]
ax.boxplot(data, orientation="horizontal", widths=0.5)
ax.set_yticks([1, 2], list(CAT.values()))
ax.set_xlabel("Fiyat (bin TL)")
ax.set_title("Fiyat dağılımı (KDV dahil mağaza fiyatı)")
fig.tight_layout()
fig.savefig(FIG / "eda_fiyat.png", dpi=150)
plt.close(fig)

                  count     mean      std      min      25%      50%      75%      max
kategori                                                                              
Buzdolabı          40.0  47458.0  12466.0  29186.0  39368.0  44004.0  54565.0  87113.0
Çamaşır makinesi   40.0  34983.0  10745.0  17179.0  28100.0  33445.0  42395.0  69749.0


## Enerji sınıfı

In [4]:
classes = ["A", "B", "C", "D", "E", "F", "G", "Yok"]
counts = (products.assign(sinif=products["energy_class"].fillna("Yok"))
          .groupby(["kategori", "sinif"]).size().unstack(fill_value=0)
          .reindex(columns=classes, fill_value=0))
print(counts.to_string())
fig, ax = plt.subplots(figsize=(6, 3.2))
x = range(len(classes))
w = 0.4
for i, k in enumerate(CAT.values()):
    ax.bar([p + (i - 0.5) * w for p in x], counts.loc[k], w, label=k)
ax.set_xticks(list(x), classes)
ax.set_ylabel("Ürün sayısı")
ax.legend(frameon=False)
ax.set_title("Enerji sınıfı (AB 2021 ölçeği)")
fig.tight_layout()
fig.savefig(FIG / "eda_enerji_sinifi.png", dpi=150)
plt.close(fig)

sinif              A  B  C  D   E  F  G  Yok
kategori                                    
Buzdolabı          0  0  1  4  28  6  0    1
Çamaşır makinesi  36  1  2  0   0  0  0    1


## Yorumlar

In [5]:
per_product = reviews.groupby(["kategori", "product_id"]).size()
print("Ürün başına yorum:")
print(per_product.groupby("kategori").describe().round(1).to_string())
top3 = reviews["product_id"].value_counts().head(3).sum() / len(reviews)
print()
print(f"En çok yorumlu 3 ürünün payı: %{top3 * 100:.0f}")
print("Site:", reviews["source"].value_counts().to_dict())
stars = reviews["rating"].value_counts(normalize=True).sort_index().mul(100).round(1)
print("Yıldız dağılımı (%):", stars.to_dict())
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(stars.index.astype(int).astype(str), stars.values, color="#4c78a8")
ax.set_xlabel("Yıldız")
ax.set_ylabel("Yorumların yüzdesi")
ax.set_title(f"Yıldız dağılımı ({len(reviews)} yorum)")
fig.tight_layout()
fig.savefig(FIG / "eda_yildiz.png", dpi=150)
plt.close(fig)

Ürün başına yorum:
                  count  mean   std  min   25%   50%   75%    max
kategori                                                         
Buzdolabı          36.0  37.9  29.1  4.0  26.8  29.0  32.5  170.0
Çamaşır makinesi   37.0  54.6  96.2  1.0  27.0  29.0  30.0  554.0

En çok yorumlu 3 ürünün payı: %28
Site: {'trendyol': 1755, 'hepsiburada': 1628}
Yıldız dağılımı (%): {1: 5.8, 2: 1.0, 3: 2.0, 4: 4.7, 5: 86.5}


## Aspect anılma sıklığı (kural tabanlı)

Kural tabanlı yöntemin bulduğu oranlar; gerçek oran değil (yöntemin doğruluğu etiketlerle ölçülecek).
Veritabanı teslimat ifadelerini ayıklayan temizlikle üretildi ("sorunsuz geldi" dayanıklılık sayılmıyor).


In [6]:
a = aspects.merge(reviews[["review_id", "kategori"]], on="review_id")
a["etiket"] = a["aspect"] + ":" + a["polarity"]
share = (a.drop_duplicates(["review_id", "etiket"]).groupby(["kategori", "etiket"]).size()
         / reviews.groupby("kategori").size()).mul(100).round(1).unstack(0).fillna(0)
print("Yorumların yüzdesi:")
print(share.to_string())
fig, ax = plt.subplots(figsize=(6, 3.6))
share = share.sort_values(list(CAT.values())[1])
y = range(len(share))
h = 0.4
for i, k in enumerate(CAT.values()):
    ax.barh([p + (i - 0.5) * h for p in y], share[k], h, label=k)
ax.set_yticks(list(y), share.index)
ax.set_xlabel("Yorumların yüzdesi")
ax.set_title("Kural tabanlı aspect anılma oranı")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG / "eda_aspect.png", dpi=150)
plt.close(fig)

Yorumların yüzdesi:
kategori        Buzdolabı  Çamaşır makinesi
etiket                                     
durability:neg        0.0               0.1
durability:pos        3.7               3.6
energy:pos            0.7               0.2
failure:neg           1.2               1.4
noise:neg             0.5               1.4
noise:pos            11.5              19.5
service:neg           0.1               0.2
service:pos           0.4               0.3
